# **Feature visualization with a library: torch-lucent and frequency parameterization**

Practice for the module [“Analysis of CNN-based Models”](https://ai-interpretability.school), lesson “Feature Visualization with a library: Practice”.

The notebook [“Feature visualization: asking the network to draw what it looks for”](https://colab.research.google.com/github/SadSabrina/open-xai-materials/blob/main/homeworks/PRACTICE_feature_visualization_en.ipynb) builds the method by hand on ResNet18: gradient ascent on the input, shifts and a penalty on pixel differences. The third trick of the lesson, replacing the image parameterization with a frequency one, it does not use. Here we take the [torch-lucent](https://github.com/greentfrapp/lucent) library, a PyTorch port of Lucid where this trick is on by default, and look at what each of its parts does:

1. one `render.render_vis` call for the same channel 12 of `layer4`;
2. what the frequency parameterization with color decorrelation gives and what the transformations give;
3. comparison with real photos: activation and pixel differences;
4. other objectives: a neuron, a class logit, diversity;
5. where it breaks: dependence on the seed and on the image size.

The notebook runs on CPU; the whole run takes two to three minutes.

Cat photo: [Paul Hanaoka, Unsplash](https://unsplash.com/photos/w2DsS-ZAP4U), Unsplash License.

Piggy photo: [Brett Sayles, Pexels](https://www.pexels.com/photo/4966539/), Pexels License.

Cat and pug photo: [Priscilla Du Preez, Unsplash](https://unsplash.com/photos/qtkpOxCpl0Y), Unsplash License.

Cyclist photo: [Tim Foster, Unsplash](https://unsplash.com/photos/wX64_Gjbclg), Unsplash License.

In [ ]:
!pip install -q torch-lucent==0.1.8   # last release May 2021; check compatibility with your torch

In [ ]:
import io
import urllib.request
import warnings
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from lucent.optvis import render, param, objectives

warnings.filterwarnings('ignore', category=UserWarning)   # kornia 0.4 and torchvision warn about deprecated arguments
SIZE = 128        # image side
STEPS = 256       # optimization steps
SEED = 0

model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).eval()
for p in model.parameters():
    p.requires_grad_(False)

## 1. One call

In lucent the objective is a string `"<layer>:<channel>"`, where the layer is the module name as in `model.named_modules()` with an underscore instead of a dot (`layer4`, `layer3_1_conv2`). `render_vis` parameterizes the image itself, adds transformations itself (shifts, scaling, rotation), normalizes the input for ImageNet itself, stretches an image with a side below 224 to 224 and returns the images at the steps listed in `thresholds` as arrays `(1, H, W, 3)` with values in $[0,1]$.

The function `vis` below fixes the seed before every call: otherwise the starting noise and the random transformations differ on every run.

In [ ]:
def vis(objective, param_f=None, transforms_=None, seed=SEED, steps=STEPS):
    torch.manual_seed(seed)
    np.random.seed(seed)
    imgs = render.render_vis(model, objective, param_f=param_f, transforms=transforms_, thresholds=(steps,),
                             show_image=False, progress=False)
    return imgs[0]


def show_row(images, titles, size=3):
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size + 0.4))
    for ax, im, t in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(im[0])
        ax.set_title(t, fontsize=9)
        ax.axis('off')
    plt.show()


lucent_default = vis('layer4:12', param_f=lambda: param.image(SIZE))
show_row([lucent_default], ['lucent, defaults: layer4:12'], size=4)

## 2. What the result is made of

`param.image(SIZE)` by default turns on two tricks:

- **frequency parameterization** (`fft=True`): the optimization runs not over pixels but over the Fourier coefficients of the image, with high frequencies scaled down. A gradient step on the coefficients changes the image smoothly, and the high-frequency noise that an adversarial example consists of gets a small weight;
- **color decorrelation** (`decorrelate=True`): the optimization runs not over the RGB channels but over their decorrelated combinations computed on ImageNet.

Separately, `render_vis` adds random **transformations** at every step: shifts, scaling, rotations; `transforms_=[]` turns them off.

**Task 1. Get four images for `layer4:12`: pixel parameterization without transformations (`param.image(SIZE, fft=False, decorrelate=False)` and `transforms_=[]`), pixel with the default transformations, frequency with decorrelation without transformations, and frequency with decorrelation with transformations (that is `lucent_default`). Plot them in a row.**

In [ ]:
# Your code here

## 3. Comparison with real photos

As in the notebook with the hand version, we compare the channel's activation on the drawn images with the activation on four real photos and compute the mean difference between neighboring pixels, a measure of “noisiness”. The activation is measured the same way for all images and the way the optimization saw it: `render_vis` stretches an image with a side below 224 to 224 (bilinearly, `align_corners=True`) and normalizes it for ImageNet, and we do the same with all images, including the photos reduced to 128. The activation is the mean over the map of channel 12 of the `layer4` output.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
captured = {}
model.layer4.register_forward_hook(lambda m, i, o: captured.__setitem__('a', o))


def activation(img, channel=12):
    # img is an array (1, H, W, 3) in [0, 1], as render_vis returns it
    t = torch.tensor(img, dtype=torch.float32).permute(0, 3, 1, 2)
    t = F.interpolate(t, size=(224, 224), mode='bilinear', align_corners=True)   # this is how render_vis stretches images below 224
    with torch.no_grad():
        model((t - MEAN) / STD)
    return captured['a'][0, channel].mean().item()


def roughness(img):
    return float(np.abs(np.diff(img, axis=1)).mean() + np.abs(np.diff(img, axis=2)).mean())


FILES = ['data/cat.jpg', 'data/piglet.jpg', 'data/cat_and_dog.jpg', 'assets/tim-foster-w-X64-Gjbclg-unsplash.jpg']
to_arr = transforms.Compose([transforms.Resize((SIZE, SIZE)), transforms.ToTensor()])
real = {}
for path in FILES:
    raw = urllib.request.urlopen('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/' + path, timeout=30).read()
    real[path.split('/')[-1]] = to_arr(Image.open(io.BytesIO(raw)).convert('RGB')).permute(1, 2, 0).numpy()[None]

**Task 2. For the four images of task 1 and the four real photos print the activation of channel 12 and the mean difference between neighboring pixels. How many times higher is the activation of each drawn image than that of the best of the four photos?**

In [ ]:
# Your code here

## 4. Other objectives

The string `"layer4:12"` is a shorthand for `objectives.channel("layer4", 12)`. Objectives in lucent add up and multiply by numbers like loss functions:

- `objectives.neuron("layer4", 12)` is one position at the center of the channel map rather than the mean over the map;
- `"fc:<class>"` is a class logit; below, ImageNet class 341, `hog` (a pig, as in the photo `piglet.jpg`);
- `objectives.channel(...) - 1e2 * objectives.diversity("layer4")` for a batch of several images is a penalty for the images of the batch being similar to each other by the layer's Gram matrices: the optimization looks for several different ways to switch the channel on.

In [ ]:
neuron_img = vis(objectives.neuron('layer4', 12), param_f=lambda: param.image(SIZE))
class_img = vis('fc:341', param_f=lambda: param.image(SIZE))
show_row([lucent_default, neuron_img, class_img],
         ['channel layer4:12 (mean over map)', 'neuron layer4:12 (center position)', 'class logit fc:341 (hog)'])

diverse = vis(objectives.channel('layer4', 12) - 1e2 * objectives.diversity('layer4'),
              param_f=lambda: param.image(SIZE, batch=4))
fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
for k, ax in enumerate(axes):
    ax.imshow(diverse[k])
    ax.set_title(f'diversity batch, image {k}: {activation(diverse[k:k + 1]):.1f}', fontsize=9)
    ax.axis('off')
plt.show()

The diversity penalty with weight 100 outweighed the objective: for three of the four images of the batch the channel barely switches on, and the images show what they can differ in rather than what the channel looks for. The penalty weight is a hyperparameter just like the weight of the pixel-difference penalty in the hand version; it is tuned by looking at the activation of every image of the batch.

## 5. Where it breaks: the seed and the image size

The lesson stresses: the problem is non-convex, and gradient ascent reaches the maximum closest to the start. The authors of the original work saw qualitatively the same picture from nine starts; we check how true this is here.

**Task 3. Get the `lucent_default` image of channel `layer4:12` for seeds 0–5. Plot the six images in a row with the activation in the title and compute the mean pairwise pixel correlation of the six images. Then repeat for seed 0 with side 64 and 224 instead of 128 (via `param.image(64)` and `param.image(224)`) and compare the images of the three sizes.**

In [ ]:
# Your code here

**Task 4. The mean pixel correlation of images from different seeds is low (about 0.13), while the activations are close. Does this contradict the authors' observation of “qualitatively the same maximum”? What should then be taken as the method's answer?**

**Your answer:**

## What to take away

- **Frequency parameterization is the third trick of the lesson, and lucent has it on by default.** Without it and without transformations, the optimization again gives a high-frequency pattern with an activation 107 times that of the best of the four photos.
- **Transformations and parameterization do different things.** Frequency parameterization lowers the pixel differences (from 0.29 to 0.16); transformations lower the activation 3–4 times and keep only a pattern robust to shifts. Even with both tricks the activation is 39 times that of the best photo: the drawing lies outside the distribution of photos.
- **The objective and its weights are also a choice.** A channel, a neuron and a class logit give different images, and a diversity penalty with weight 100 outweighs the objective itself: three of the four images of the batch do not switch the channel on.
- **The image depends on the seed and the size.** The pixels of different runs do not coincide; what recurs is the texture, not the composition.
- **torch-lucent is a port of Lucid; its last release, 0.1.8, came out in May 2021.** Here it works with Colab's torch 2.11 but relies on kornia 0.4; check signatures and compatibility before use.

Sources:
1. [Olah, Mordvintsev & Schubert, 2017. Feature visualization](https://distill.pub/2017/feature-visualization/)
2. [Erhan et al., 2009. Visualizing higher-layer features of a deep network](https://www.researchgate.net/publication/265022827_Visualizing_Higher-Layer_Features_of_a_Deep_Network)
3. [torch-lucent](https://github.com/greentfrapp/lucent), [Lucid](https://github.com/tensorflow/lucid)